# Lab 8 - a two-layer network in NumPy

**Session 8.** Build the network, run the tiny-batch correctness check FIRST, then train
properly with early stopping and compare against logistic regression on the same split.

## 1. Data

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_classification(n_samples=1500, n_features=10, n_informative=6,
                           class_sep=0.7, random_state=2026)
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.4, stratify=y,
                                              random_state=2026)
X_va, X_te, y_va, y_te = train_test_split(X_rest, y_rest, test_size=0.5,
                                          stratify=y_rest, random_state=2026)

sc = StandardScaler().fit(X_tr)          # fitted on TRAIN only
X_tr, X_va, X_te = sc.transform(X_tr), sc.transform(X_va), sc.transform(X_te)
print(X_tr.shape, X_va.shape, X_te.shape)

## 2. The network

Forward, backward, update - the three functions from the lecture, vectorised over rows.

In [ ]:
def init_params(n_features, n_hidden, seed=0):
    rng = np.random.default_rng(seed)
    return {
        # He initialisation: variance 2/fan_in, which keeps ReLU activations alive
        "W1": rng.normal(0, np.sqrt(2 / n_features), (n_features, n_hidden)),
        "b1": np.zeros(n_hidden),
        "W2": rng.normal(0, np.sqrt(2 / n_hidden), n_hidden),
        "b2": 0.0,
    }


def forward(X, p):
    z1 = X @ p["W1"] + p["b1"]
    h = np.maximum(0.0, z1)                       # ReLU
    z2 = h @ p["W2"] + p["b2"]
    y_hat = 1.0 / (1.0 + np.exp(-z2))             # sigmoid
    return z1, h, y_hat


def log_loss(y, y_hat, eps=1e-12):
    y_hat = np.clip(y_hat, eps, 1 - eps)
    return float(-np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat)))


def backward(X, y, p, cache):
    z1, h, y_hat = cache
    n = len(y)
    d2 = (y_hat - y) / n                          # dL/dz2
    gW2, gb2 = h.T @ d2, d2.sum()
    d1 = np.outer(d2, p["W2"]) * (z1 > 0)         # chain rule through the ReLU
    return {"W1": X.T @ d1, "b1": d1.sum(0), "W2": gW2, "b2": gb2}


def step(p, grads, alpha):
    return {k: p[k] - alpha * grads[k] for k in p}

## 3. The correctness check: overfit twenty rows

If a network cannot drive the loss to near zero on twenty rows, the bug is in the code, not
the model. Run this before anything else, every time.

In [ ]:
Xs, ys = X_tr[:20], y_tr[:20].astype(float)
p = init_params(Xs.shape[1], 32, seed=0)
for i in range(4000):
    cache = forward(Xs, p)
    p = step(p, backward(Xs, ys, p, cache), alpha=0.2)
    if i % 1000 == 0:
        print(f"  iter {i:5d}  loss {log_loss(ys, cache[2]):.5f}")
print(f"final loss on 20 rows: {log_loss(ys, forward(Xs, p)[2]):.6f}  "
      f"(should be ~0 - if not, debug before proceeding)")

## 4. Train with mini-batches and early stopping

In [ ]:
def train(X, y, X_va, y_va, n_hidden=32, alpha=0.1, batch=64,
          epochs=200, patience=10, seed=0):
    rng = np.random.default_rng(seed)
    p = init_params(X.shape[1], n_hidden, seed)
    best, best_p, wait, history = np.inf, p, 0, []
    for _ in range(epochs):
        order = rng.permutation(len(y))
        for s in range(0, len(y), batch):
            idx = order[s:s + batch]
            cache = forward(X[idx], p)
            p = step(p, backward(X[idx], y[idx], p, cache), alpha)
        tr = log_loss(y, forward(X, p)[2])
        va = log_loss(y_va, forward(X_va, p)[2])
        history.append((tr, va))
        if va < best - 1e-5:
            best, best_p, wait = va, p, 0
        else:
            wait += 1
            if wait >= patience:
                break                     # early stopping: validation loss stopped improving
    return best_p, np.array(history)


p, hist = train(X_tr, y_tr.astype(float), X_va, y_va.astype(float))
print(f"stopped after {len(hist)} epochs; best validation loss {hist[:, 1].min():.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(hist[:, 0], label="train")
ax.plot(hist[:, 1], label="validation")
ax.axvline(hist[:, 1].argmin(), ls="--", color="grey", label="early stop")
ax.set(xlabel="epoch", ylabel="log-loss", title="Training curve")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Against logistic regression, same split

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

p_te = forward(X_te, p)[2]
lr = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
p_lr = lr.predict_proba(X_te)[:, 1]

print(f"network  test accuracy {accuracy_score(y_te, p_te >= 0.5):.3f}   "
      f"AUC {roc_auc_score(y_te, p_te):.3f}")
print(f"logistic test accuracy {accuracy_score(y_te, p_lr >= 0.5):.3f}   "
      f"AUC {roc_auc_score(y_te, p_lr):.3f}")
print("\nOn tabular data with a near-linear boundary, the extra machinery often buys little.")
print("Bring a network when you have a reason - not because it is the fashionable answer.")

## Exercises

1. **Width and depth.** Try `n_hidden` in `[2, 8, 32, 128]`. Where does test AUC stop
   improving, and where does the training curve start to diverge from validation?
2. **Kill the non-linearity.** Replace `np.maximum(0.0, z1)` with `z1` (the identity) and
   retrain. Explain the result: what model have you actually fitted?
3. **Learning rate.** Try `alpha` in `[0.001, 0.01, 0.1, 1.0]` and plot the four training
   curves together. Which fails, and in which direction?